# Notebook 37 · WeatherNext 2 JPCZ ensemble verification

Compare **all 64 members** with ERA5 for **925-hPa horizontal divergence** in the offshore
JPCZ polygon and coastal wedge, at **1, 5, and 10 days**. Negative values indicate
convergence. The two polygons are evaluated independently, including their overlap.

Start in **Pilot** mode. It runs one catalog event and reports six ERA5 ranks. Then switch to
**Catalog** and rerun sections 3–5. Synced Drive checkpoints allow each completed ERA5
target and forecast lead to be reused after a runtime disconnect.

The catalog is the same CSV Notebook 35 reads. Notebook 36 supplies the archive reader and
onset timing rule. You can run this notebook directly using that CSV and your approved
WeatherNext account. The plotting batches in Notebooks 35 and 36 are optional.

[WeatherNext 2 data and attribution](https://developers.google.com/earth-engine/datasets/catalog/projects_gcp-public-data-weathernext_assets_weathernext_2_0_0)
· [Rank histogram method, Gneiting et al. (2007)](https://sites.stat.washington.edu/raftery/Research/PDF/Gneiting2007jrssb.pdf)
· [Interpretation and case selection, Hamill (2001)](https://doi.org/10.1175/1520-0493%282001%29129%3C0550%3AIORHFV%3E2.0.CO%3B2)


In [ ]:
# @title 1 · Install dependencies in a fresh CPU runtime
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy>=1.26,<2.4', 'pandas>=2.2,<3', 'xarray>=2026.4,<2027',
                'zarr>=3.1,<4', 'gcsfs>=2025.3,<2027', 'dask[array]>=2025.1,<2027',
                'matplotlib>=3.9,<3.11', 'xlsxwriter>=3.2,<4'], check=True)


In [ ]:
# @title 2 · Connect Google Drive and authenticate for WeatherNext
from google.colab import auth, drive

drive.mount('/content/drive')
auth.authenticate_user()
import google.auth
credentials, detected_project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/cloud-platform'])
print('Drive mounted and credentials acquired. Use the approved account for WN2.')


## 3 · Choose the cases

The study dates match Notebook 36’s provisional 2023-onward window by default. The
operational model’s training cutoff has not been verified. Pilot event **0** selects the
first event in the study window. You can enter an event number that worked in Notebook 36
instead. The inventory displayed below includes the numbers and timing offsets.

The target is the nearest 00, 06, 12, or 18 UTC on the original onset’s UTC date, with
ties going earlier. ERA5 is evaluated at that exact time. The catalog’s peak D12 is kept
as context. If required by your Cloud account, enter your existing project ID.


In [ ]:
# @title 3a · Settings
from pathlib import Path

RUN_MODE = 'Pilot'  # @param ["Pilot", "Catalog"]
PILOT_EVENT_NUMBER = 0  # @param {type:"integer"}
PROJECT_ID = ''  # @param {type:"string"}
DRIVE_ROOT = Path('/content/drive/MyDrive/JPCZcatalog_outputs')
CATALOG_PATH = DRIVE_ROOT / 'jpcz_catalog_ndjf_merged_12h.csv'
NOTEBOOK35_DIR = DRIVE_ROOT / 'imerg_manual_case_review'
OUTPUT_DIR = DRIVE_ROOT / 'weathernext2_ensemble_verification'
STUDY_START_UTC = '2023-01-01'  # @param {type:"date"}
STUDY_END_UTC = ''  # @param {type:"string"} # Optional exclusive UTC date.
RUN_OVERRIDES = {}


In [ ]:
# @title 3b · Load Notebook 36 archive helpers (run unchanged)
"""WeatherNext 2 single-member atlas, also embedded in standalone Notebook 36.

No credentials, remote listings, or forecast fields are loaded at import time.
Build the notebook with tools/build_weathernext_notebook.py after editing this file.
"""
from __future__ import annotations

from dataclasses import dataclass, replace
from pathlib import Path
import hashlib
import html
import json
import re

import numpy as np
import pandas as pd
import xarray as xr


LEAD_HOURS = (24, 120, 240)
DATE_COLUMNS = ("event_start", "event_end", "event_peak")
ARCHIVE_ROOT = "weathernext/weathernext_2_0_0/zarr"
SCHEMA_VERSION = 2
PLOT_STYLE_VERSION = 4
VORTEX_LABEL_SCHEME = "notebook35_vortex_screen_v3"
WN2_PRESSURE_LEVELS = (50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000)
EARTH_RADIUS_M = 6371000.0
EARTH_OMEGA_S1 = 7.292115e-5
STANDARD_GRAVITY = 9.80665
KAPPA = 2.0 / 7.0
DERIVED_FIELDS = {"divergence", "ertel_pv"}
# The historical merged stores omit variable-level units. Use the published WN2
# schema only for known variables with ABSENT units; never override declared units.
WN2_SCHEMA_UNITS = {
    "geopotential": "m2 s-2", "u_component_of_wind": "m s-1",
    "v_component_of_wind": "m s-1", "temperature": "K",
    "specific_humidity": "kg kg-1", "vertical_velocity": "Pa s-1",
    "mean_sea_level_pressure": "Pa", "2m_temperature": "K",
    "sea_surface_temperature": "K", "total_precipitation_6hr": "m",
    "10m_u_component_of_wind": "m s-1", "10m_v_component_of_wind": "m s-1",
    "100m_u_component_of_wind": "m s-1", "100m_v_component_of_wind": "m s-1",
}
POLYGONS = {
    "JPCZ polygon": [(129.5, 41.0), (136.0, 37.4), (134.5, 35.8), (128.8, 38.0)],
    "Coastal wedge": [(133.05, 35.55), (136.05, 35.55), (139.55, 39.00), (139.55, 42.55)],
}
DOMAINS = {
    "Circulation (110–160°E, 25–60°N)": (110., 160., 25., 60.),
    "Notebook 35 region (125–145°E, 33–47°N)": (125., 145., 33., 47.),
}


@dataclass(frozen=True)
class FieldSpec:
    key: str
    label: str
    variables: tuple[str, ...]
    units: str
    palette: str
    limits: tuple[float, float]
    level: int | None = None
    conversion: str = "identity"

    @property
    def title(self):
        return f"{self.level}-hPa {self.label}" if self.level is not None else self.label


FIELD_PRESETS = {
    "height": FieldSpec("height", "geopotential height", ("geopotential",), "m", "viridis", (4800, 6000), 500, "height"),
    "winds": FieldSpec("winds", "winds", ("u_component_of_wind", "v_component_of_wind"), "m s⁻¹", "YlGnBu", (0, 30), 925, "wind"),
    "divergence": FieldSpec("divergence", "horizontal divergence (derived)", ("u_component_of_wind", "v_component_of_wind"), "10⁻⁵ s⁻¹", "RdBu_r", (-10, 10), 925, "divergence"),
    "ertel_pv": FieldSpec("ertel_pv", "Ertel PV (derived)", ("temperature", "u_component_of_wind", "v_component_of_wind"), "PVU", "RdBu_r", (-2, 2), 1000, "ertel_pv"),
    "temperature": FieldSpec("temperature", "temperature", ("temperature",), "°C", "coolwarm", (-40, 20), 850, "temperature"),
    "humidity": FieldSpec("humidity", "specific humidity", ("specific_humidity",), "g kg⁻¹", "YlGnBu", (0, 10), 925, "humidity"),
    "vertical_velocity": FieldSpec("vertical_velocity", "vertical velocity", ("vertical_velocity",), "Pa s⁻¹", "RdBu_r", (-1, 1), 700),
    "mslp": FieldSpec("mslp", "mean sea-level pressure", ("mean_sea_level_pressure",), "hPa", "viridis", (980, 1040), None, "pressure"),
    "t2m": FieldSpec("t2m", "2-m temperature", ("2m_temperature",), "°C", "coolwarm", (-20, 20), None, "temperature"),
    "winds10m": FieldSpec("winds10m", "10-m winds", ("10m_u_component_of_wind", "10m_v_component_of_wind"), "m s⁻¹", "YlGnBu", (0, 25), None, "wind"),
    "precipitation": FieldSpec("precipitation", "preceding 6-h precipitation", ("total_precipitation_6hr",), "mm / 6 h", "YlGnBu", (0, 30), None, "precipitation"),
}


def utc_naive(value):
    """Use timezone-free UTC for xarray; never interpret a timestamp as local time."""
    stamp = pd.Timestamp(value)
    if pd.isna(stamp):
        raise ValueError("Missing UTC timestamp.")
    return stamp.tz_localize("UTC").tz_localize(None) if stamp.tzinfo is None else stamp.tz_convert("UTC").tz_localize(None)


def iso_utc(value):
    return utc_naive(value).isoformat() + "Z"


def vortex_classification(record):
    """Use completed event-screen results; legacy manual case labels are unrelated."""
    status = str(record.get("vortex_scan_status", record.get("status", "not_scanned"))).strip().lower()
    if status == "ambiguous":
        return "Vortex status ambiguous"
    if status in {"unavailable", "failed", "error"}:
        return "Vortex status unavailable"
    if status != "complete":
        return "Vortex not checked"
    value = record.get("vortex_detected")
    if isinstance(value, (bool, np.bool_)):
        detected = bool(value)
    elif isinstance(value, (int, float, np.integer, np.floating)) and pd.notna(value) and value in (0, 1):
        detected = bool(value)
    elif isinstance(value, str) and value.strip().lower() in {"true", "false", "0", "1"}:
        detected = value.strip().lower() in {"true", "1"}
    else:
        return "Vortex status unknown"
    return "Vortex" if detected else "Non-vortex"


def vortex_plot_label(record):
    """Use the positive association flag for the two labels shown on figures."""
    if vortex_classification(record) == "Vortex":
        return "Vortex-associated"
    return "Non-vortex-associated"


def vortex_label_metadata(record):
    label = vortex_classification(record)
    detected = True if label == "Vortex" else False if label == "Non-vortex" else None
    return {"vortex_classification": label, "vortex_plot_label": vortex_plot_label(record),
            "vortex_scan_status": str(record.get("vortex_scan_status", "not_scanned")),
            "vortex_detected": detected, "vortex_label_scheme": VORTEX_LABEL_SCHEME,
            "vortex_label_source": str(record.get("vortex_label_source", "saved Notebook 35 event metadata")),
            "vortex_screen_signature": str(record.get("vortex_screen_signature", ""))}


def refresh_vortex_labels(frame, review_dir):
    """Mirror Notebook 35's per-event JSON cache under its active screen signature.

    The inventory CSV can predate a scan. The JSON event records are what Notebook
    35 uses in its event dropdown, so they take precedence, without rerunning scans.
    """
    frame, root = frame.copy(), Path(review_dir) / "vortex_screen"
    active = root / "active_screen.json"
    folder, issue = None, None
    if active.exists():
        try:
            signature = json.loads(active.read_text())["signature"]
            if not isinstance(signature, str) or not re.fullmatch(r"[0-9a-f]{16}", signature):
                raise ValueError("Invalid active vortex-screen signature.")
            folder = root / signature
        except (ValueError, KeyError, OSError, TypeError):
            issue = "unavailable"
    else:
        folders = [p for p in root.glob("*") if p.is_dir() and re.fullmatch(r"[0-9a-f]{16}", p.name)]
        if len(folders) == 1:
            folder = folders[0]
        elif len(folders) > 1:
            issue = "ambiguous"
    records = []
    for _, row in frame.iterrows():
        record = row.to_dict()
        if issue:
            record.update(vortex_scan_status=issue, vortex_detected=None,
                          vortex_label_source="Refresh catalog in updated Notebook 35 to identify its active screen.")
        elif folder is not None:
            path = folder / ("event_" + hashlib.sha256(str(row.review_key).encode()).hexdigest()[:20] + ".json")
            record.update(vortex_scan_status="not_scanned", vortex_detected=None,
                          vortex_label_source=str(path), vortex_screen_signature=folder.name)
            if path.exists():
                try:
                    saved = json.loads(path.read_text())
                    if (not isinstance(saved, dict) or saved.get("status") != "complete" or saved.get("review_key") != row.review_key
                            or saved.get("signature") != folder.name):
                        raise ValueError("Vortex record identity, signature, or completion status does not match.")
                    record.update(vortex_scan_status="complete", vortex_detected=saved.get("vortex_detected"))
                except (ValueError, OSError):
                    record.update(vortex_scan_status="unavailable", vortex_detected=None)
        else:
            record["vortex_label_source"] = "saved Notebook 35 inventory; no event cache folder found"
        records.append(vortex_label_metadata(record))
    updates = pd.DataFrame(records, index=frame.index)
    for column in updates:
        frame[column] = updates[column]
    return frame


def nearest_start_date_time(start):
    """Nearest 00/06/12/18 on the SAME UTC date; equal distances choose earlier."""
    start = utc_naive(start)
    choices = [start.normalize() + pd.Timedelta(hours=h) for h in (0, 6, 12, 18)]
    return min(choices, key=lambda value: (abs(value - start), value))


def normalise_catalog(frame):
    frame = frame.copy()
    for column in DATE_COLUMNS:
        if column not in frame:
            raise ValueError(f"Catalog is missing {column}; use Notebook 35's current merged catalog.")
        frame[column] = pd.to_datetime(frame[column], utc=True, errors="raise").dt.tz_localize(None)
        if frame[column].isna().any():
            raise ValueError(f"Missing timestamps in {column}.")
    if ((frame.event_end < frame.event_start) | (frame.event_peak < frame.event_start)
            | (frame.event_peak > frame.event_end)).any():
        raise ValueError("Each event must satisfy start ≤ peak ≤ end.")
    frame["review_key"] = frame[list(DATE_COLUMNS)].apply(lambda row: "|".join(t.isoformat() for t in row), axis=1)
    if frame.review_key.duplicated().any():
        raise ValueError("Duplicate event start/end/peak identities in catalog.")
    frame = frame.sort_values("event_peak").reset_index(drop=True)
    numbers = pd.to_numeric(frame.get("event_number", pd.Series(index=frame.index, dtype=float)), errors="coerce")
    valid = numbers.notna().all() and numbers.gt(0).all() and numbers.eq(numbers.round()).all() and numbers.is_unique
    frame["event_number"] = numbers.astype(int) if valid else np.arange(1, len(frame) + 1)
    span = (frame.event_end - frame.event_start) / pd.Timedelta(hours=1) + 1
    duration = pd.to_numeric(frame.get("episode_span_hours", frame.get("duration_hours", span)), errors="coerce")
    frame["event_duration_hours"] = duration.where(duration.gt(0)).fillna(span)
    frame["target_valid_time"] = frame.event_start.map(nearest_start_date_time)
    frame["onset_offset_hours"] = (frame.target_valid_time - frame.event_start) / pd.Timedelta(hours=1)
    if "event_peak_D_1e5_s-1" not in frame:
        frame["event_peak_D_1e5_s-1"] = pd.to_numeric(frame.get("event_peak_D_s-1", np.nan), errors="coerce") * 1e5
    return frame


def read_catalog(catalog_path, review_dir, study_start="2023-01-01", study_end=None):
    """Read original inventory; merge reviews only on exact event windows."""
    catalog_path, review_dir = Path(catalog_path), Path(review_dir)
    if not catalog_path.exists():
        raise FileNotFoundError(f"Current catalog not found: {catalog_path}. Check the Drive path used in Notebook 35.")
    catalog = normalise_catalog(pd.read_csv(catalog_path))
    raw_path = catalog_path.parent / "jpcz_catalog_ndjf.csv"
    if raw_path.exists():
        raw = normalise_catalog(pd.read_csv(raw_path))
        missing = [r.event_start for r in raw.itertuples() if not
                   ((catalog.event_start <= r.event_start) & (catalog.event_end >= r.event_end)).any()]
        if missing:
            raise ValueError(f"Merged catalog omits {len(missing)} saved raw fragments. Refresh it with Notebook 06, as in Notebook 35.")
    for filename in ("current_catalog_review_inventory.csv", "case_review_labels.csv"):
        path = review_dir / filename
        if not path.exists():
            continue
        saved = pd.read_csv(path)
        if all(c in saved for c in DATE_COLUMNS):
            for column in DATE_COLUMNS:
                saved[column] = pd.to_datetime(saved[column], utc=True, errors="raise").dt.tz_localize(None)
            saved["review_key"] = saved[list(DATE_COLUMNS)].apply(lambda row: "|".join(t.isoformat() for t in row), axis=1)
        if "review_key" not in saved:
            continue
        columns = [c for c in saved if c.startswith("historical_") or c in
                   ("case_label", "case_notes", "vortex_detected", "vortex_scan_status")]
        if saved.review_key.duplicated().any():
            raise ValueError(f"Duplicate event identities in {path}; resolve before joining reviews.")
        saved = saved.set_index("review_key")
        for column in columns:
            mapped = catalog.review_key.map(saved[column])
            catalog[column] = mapped.combine_first(catalog[column]) if column in catalog else mapped
    catalog = refresh_vortex_labels(catalog, review_dir)
    start = utc_naive(study_start)
    included = catalog.event_start.ge(start)
    reasons = pd.Series("included", index=catalog.index)
    reasons.loc[~included] = "before configured study start (training cutoff unverified)"
    if study_end:
        after = catalog.event_start.ge(utc_naive(study_end))
        included &= ~after
        reasons.loc[after] = "at/after configured study end (exclusive)"
    report = catalog.copy()
    report["selection_status"] = reasons
    report["training_cutoff_status"] = "operational archive training cutoff not verified"
    report["forecast_availability"] = "not checked; use WN2 access check for each event"
    return catalog.loc[included].copy(), report


def forecast_requests(row):
    target = nearest_start_date_time(row["event_start"])
    return pd.DataFrame([{
        "lead_hours": lead, "lead_days": lead // 24,
        "init_time": target - pd.Timedelta(hours=lead), "valid_time": target,
        "event_start": utc_naive(row["event_start"]),
        "onset_offset_hours": (target - utc_naive(row["event_start"])) / pd.Timedelta(hours=1),
    } for lead in LEAD_HOURS])


class GoogleCloudArchive:
    """Metadata-first discovery of WN2 runs, without recursive archive downloads."""
    def __init__(self, credentials, project_id="", root=ARCHIVE_ROOT, overrides=None):
        import gcsfs
        self.root = root.removeprefix("gs://").rstrip("/")
        if self.root != ARCHIVE_ROOT:
            raise ValueError("Use the WN2 ensemble archive root; do not use /success or the mean archive.")
        self.storage_options = {"token": credentials}
        if project_id.strip():
            self.storage_options["project"] = project_id.strip()
        self.fs = gcsfs.GCSFileSystem(**self.storage_options)
        self.overrides = {iso_utc(k): str(v).removeprefix("gs://").rstrip("/") for k, v in (overrides or {}).items()}
        self.listings = {}
        self.datasets = {}
        self.resolved = {}

    def _list(self, path):
        if path not in self.listings:
            self.listings[path] = self.fs.ls(path, detail=False)
        return self.listings[path]

    def _is_store(self, path):
        return self.fs.exists(path + "/zarr.json") or self.fs.exists(path + "/.zgroup")

    def resolve(self, init_time):
        init_time = utc_naive(init_time)
        identity = iso_utc(init_time)
        if identity in self.resolved:
            return self.resolved[identity]
        if identity in self.overrides:
            candidates = [self.overrides[identity]]
        else:
            partitions = []
            for path in self._list(self.root):
                match = re.fullmatch(r"(\d{4})_to_(\d{4}|present)", path.rstrip("/").rsplit("/", 1)[-1])
                if match and int(match[1]) <= init_time.year <= (9999 if match[2] == "present" else int(match[2])):
                    partitions.append(path.rstrip("/"))
            prefix = init_time.strftime("%Y%m%d_%Hhr")
            candidates = []
            for partition in partitions:
                historical = partition + "/predictions.zarr"
                if self._is_store(historical):
                    # Labels such as 2022_to_2023 use an exclusive upper boundary.
                    # Inspect the actual initialization coordinate, not the label.
                    if contains_initialization(self.open(historical), init_time):
                        candidates.append(historical)
                    continue
                for path in self._list(partition):
                    name = path.rstrip("/").rsplit("/", 1)[-1]
                    if name.startswith(prefix + "_") or name == prefix:
                        candidates.append(path.rstrip("/") + "/predictions.zarr")
        stores = sorted({path for path in candidates if self._is_store(path)})
        if not stores:
            raise FileNotFoundError(f"No WN2 store contains exact initialization {identity}. "
                                    "Checked historical period stores and dated run folders; no other run was substituted.")
        if len(stores) != 1:
            raise ValueError(f"Multiple WN2 stores for {identity}: {stores}. Set RUN_OVERRIDES to an explicit store.")
        if not stores[0].startswith(self.root + "/"):
            raise ValueError("Run overrides must remain within the WN2 ensemble archive.")
        self.resolved[identity] = stores[0]
        return stores[0]

    def open(self, store):
        if store not in self.datasets:
            # xarray/zarr construct an async-capable fsspec store from this URL.
            ds = xr.open_zarr("gs://" + store, storage_options=self.storage_options,
                              chunks={}, consolidated=None)
            if "sample" not in ds.coords or "sample" not in ds.dims:
                ds.close()
                raise ValueError("Expected the member-resolved WN2 sample coordinate, not ensemble-mean data.")
            # Keep only a few metadata handles; fields remain lazy.
            if len(self.datasets) >= 4:
                self.datasets.pop(next(iter(self.datasets))).close()
            self.datasets[store] = ds
        return self.datasets[store]

    def run(self, init_time):
        store = self.resolve(init_time)
        return store, canonical_run(self.open(store), init_time)

    def clear(self):
        for ds in self.datasets.values():
            ds.close()
        self.datasets.clear()
        self.listings.clear()
        self.resolved.clear()
        self.fs.invalidate_cache()


def contains_initialization(ds, init_time):
    init_time = utc_naive(init_time)
    if "prediction_timedelta" in ds.dims and "time" in ds.coords:
        if not np.issubdtype(ds.time.dtype, np.datetime64):
            raise ValueError("Historical WN2 time must decode to initialization datetimes.")
        return np.datetime64(init_time, "ns") in ds.time.values
    if "init_time" in ds.coords and ds.init_time.size == 1:
        return utc_naive(ds.init_time.values.reshape(-1)[0]) == init_time
    raise ValueError("Unrecognized WN2 initialization-coordinate layout.")


def canonical_run(ds, init_time):
    """Normalize historical and per-run layouts without loading forecast fields."""
    init_time = utc_naive(init_time)
    if not contains_initialization(ds, init_time):
        raise ValueError(f"Store does not contain exact initialization {iso_utc(init_time)}.")
    if "prediction_timedelta" in ds.dims:
        run = ds.sel(time=np.datetime64(init_time, "ns"), drop=True)
        if not np.issubdtype(run.prediction_timedelta.dtype, np.timedelta64):
            raise ValueError("Historical prediction_timedelta must decode to timedeltas.")
        run = run.rename({"prediction_timedelta": "time"})
        if "lead_time_secs" in run.coords:
            expected = run.time.values / np.timedelta64(1, "s")
            if not np.array_equal(run.lead_time_secs.values, expected):
                raise ValueError("Historical lead_time_secs disagrees with prediction_timedelta.")
        run = run.assign_coords(init_time=np.datetime64(init_time, "ns"),
                                datetime=("time", np.datetime64(init_time, "ns") + run.time.values))
        layout = "historical_period_store"
    else:
        run, layout = ds, "dated_run_store"
    run = run.copy(deep=False)
    run.attrs = {**run.attrs, "source_layout": layout}
    for name in run.data_vars:
        attrs = dict(run[name].attrs)
        if not attrs.get("units") and name in WN2_SCHEMA_UNITS:
            attrs.update(units=WN2_SCHEMA_UNITS[name],
                         units_source="official WN2 schema; absent from source variable attributes")
        else:
            attrs["units_source"] = "source variable attributes"
        run[name].attrs = attrs
    return run


def select_forecast(ds, init_time, lead_hours, member, variables, level, domain, *, level_stencil=None):
    """Select an exact lead, level, and region, keeping all samples when member is None."""
    required = {"sample", "time", "lat", "lon", "init_time"}
    if not required.issubset(ds.coords):
        raise ValueError(f"WN2 schema missing coordinates: {sorted(required - set(ds.coords))}.")
    init_time = utc_naive(init_time)
    if ds.init_time.size != 1 or utc_naive(ds.init_time.values.reshape(-1)[0]) != init_time:
        raise ValueError("Store init_time does not match the requested initialization.")
    if not np.issubdtype(ds.time.dtype, np.timedelta64):
        raise ValueError("WN2 time must decode to forecast-lead timedeltas; refusing ambiguous time units.")
    if not (0 < lead_hours <= 360 and lead_hours % 6 == 0):
        raise ValueError("Expected a six-hour lead within the 15-day forecast horizon.")
    lead = pd.Timedelta(hours=lead_hours)
    if member is not None and member not in ds.sample.values:
        raise ValueError(f"Member label {member!r} is not available in this run.")
    if np.timedelta64(lead.value, "ns") not in ds.time.values:
        raise ValueError(f"Exact {lead_hours}-hour lead is unavailable; no nearest forecast was substituted.")
    missing = set(variables) - set(ds.data_vars)
    if missing:
        raise ValueError(f"Requested fields unavailable: {sorted(missing)}.")
    snap = ds[list(variables)].sel(time=lead)
    if member is not None:
        snap = snap.sel(sample=member)
    target = init_time + lead
    if "datetime" in snap.coords and (snap.datetime.size != 1 or utc_naive(snap.datetime.values.reshape(-1)[0]) != target):
        raise ValueError("WN2 datetime is inconsistent with initialization + lead.")
    if level is not None:
        if "level" not in snap.dims or level not in snap.level.values:
            raise ValueError(f"Pressure level {level} hPa is unavailable.")
        if level_stencil is not None:
            if len(level_stencil) != 3 or len(set(level_stencil)) != 3 or level not in level_stencil:
                raise ValueError("PV needs three distinct pressure levels including the target.")
            if not set(level_stencil).issubset(set(snap.level.values)):
                raise ValueError("Requested PV pressure stencil is unavailable.")
            snap = snap.sel(level=list(level_stencil)).sortby("level")
        else:
            snap = snap.sel(level=level)
    elif "level" in snap.dims:
        raise ValueError("Surface preset unexpectedly retains a pressure-level dimension.")
    west, east, south, north = domain
    if not (-180 <= west < east <= 360 and -90 <= south < north <= 90):
        raise ValueError("Invalid geographic domain.")
    # The atlas domains are in East Asia, valid in both common longitude conventions.
    for coordinate, lo, hi in (("lat", south, north), ("lon", west, east)):
        values = np.asarray(snap[coordinate].values)
        if values.ndim != 1 or len(values) < 2:
            raise ValueError(f"Expected a one-dimensional {coordinate} grid.")
        snap = snap.sel({coordinate: slice(lo, hi) if values[0] < values[-1] else slice(hi, lo)})
        if snap.sizes[coordinate] < 2:
            raise ValueError(f"Requested region has too few {coordinate} cells.")
        actual = snap[coordinate].values
        if float(np.min(actual)) > lo + .26 or float(np.max(actual)) < hi - .26:
            raise ValueError(f"Forecast grid does not cover requested {coordinate} bounds.")
    sample_dims = ("sample",) if member is None else ()
    pressure_dims = ("level",) if level_stencil is not None else ()
    order = sample_dims + pressure_dims + ("lat", "lon")
    for name in variables:
        if set(snap[name].dims) != set(order):
            raise ValueError(f"Unexpected remaining dimensions for {name}: {snap[name].dims}.")
    return snap.transpose(*order).sortby("lat").sortby("lon")


def pv_pressure_stencil(available_levels, target):
    levels = sorted({int(p) for p in available_levels})
    if target not in levels or len(levels) < 3:
        raise ValueError("Ertel PV requires the target and at least three available pressure levels.")
    index = levels.index(target)
    first = max(0, min(index - 1, len(levels) - 3))
    stencil = levels[first:first + 3]
    method = "centered" if stencil[0] < target < stencil[-1] else "one-sided at pressure-range boundary"
    return stencil, method


def diagnostic_settings(ds, spec, domain):
    if spec.key not in DERIVED_FIELDS:
        return None
    settings = {"algorithm_version": 2, "numerical_precision": "float64", "earth_radius_m": EARTH_RADIUS_M,
                "horizontal_derivative": "second-order differences on spherical latitude/longitude grid",
                "halo_cells": 2, "below_ground_mask": "not applied; source surface pressure/terrain unavailable",
                "temporal_operation": "instantaneous forecast snapshot"}
    if spec.key == "ertel_pv":
        stencil, method = pv_pressure_stencil(ds.level.values, spec.level)
        settings.update(pressure_stencil_hpa=stencil, vertical_derivative=method,
                        pressure_derivative_units="Pa", theta_reference_hpa=1000,
                        theta_kappa=KAPPA, earth_omega_s1=EARTH_OMEGA_S1,
                        gravity_ms2=STANDARD_GRAVITY, pv_definition="dry hydrostatic pressure-coordinate Ertel PV",
                        formula="-g * (du_dp*dtheta_dy - dv_dp*dtheta_dx + (zeta+f)*dtheta_dp)",
                        display_multiplier=1e6)
    else:
        settings.update(formula="[du/dlambda + d(v*cos(phi))/dphi] / [R*cos(phi)]", display_multiplier=1e5)
    return settings


def diagnostic_domain(ds, domain):
    """Include two source cells outside map bounds when available, then crop after differentiation."""
    lat, lon = ds.lat.values, ds.lon.values
    dlat, dlon = abs(float(lat[1] - lat[0])), abs(float(lon[1] - lon[0]))
    west, east, south, north = domain
    return (max(west - 2 * dlon, float(np.min(lon))), min(east + 2 * dlon, float(np.max(lon))),
            max(south - 2 * dlat, float(np.min(lat))), min(north + 2 * dlat, float(np.max(lat))))


def derive_field(snap, spec, domain):
    """Dry hydrostatic PV and signed horizontal divergence with spherical metric terms.

    u/v are eastward/northward winds. Pressure is in Pa; theta references 1000 hPa.
    PV includes both vertical-shear/horizontal-theta terms, not just (zeta+f)*dtheta/dp.
    """
    for name in spec.variables:
        expected = {"kelvin"} if name == "temperature" else {"wind"}
        if unit_kind(snap[name].attrs.get("units", "")) not in expected:
            raise ValueError(f"Unexpected units for derived-field input {name}: {snap[name].attrs.get('units')!r}.")
    # Only regional inputs are retained; original packed source chunks are decoded serially.
    snap = snap.load(scheduler="synchronous")
    phi = np.deg2rad(np.asarray(snap.lat.values, dtype=float))
    lam = np.deg2rad(np.asarray(snap.lon.values, dtype=float))
    if len(phi) < 3 or len(lam) < 3 or np.any(np.abs(np.cos(phi)) < 1e-6):
        raise ValueError("Diagnostics need at least three latitudes/longitudes away from the poles.")
    if np.any(np.diff(phi) <= 0) or np.any(np.diff(lam) <= 0):
        raise ValueError("Diagnostic latitude/longitude must increase; select_forecast normalizes ordering.")
    cosphi = np.cos(phi)[:, None]
    u = np.asarray(snap.u_component_of_wind.values, dtype=float)
    v = np.asarray(snap.v_component_of_wind.values, dtype=float)
    if spec.key == "divergence":
        values = (np.gradient(u, lam, axis=-1, edge_order=2)
                  + np.gradient(v * cosphi, phi, axis=-2, edge_order=2)) / (EARTH_RADIUS_M * cosphi) * 1e5
        dims = snap.u_component_of_wind.dims
    elif spec.key == "ertel_pv":
        if "sample" in snap.dims:
            raise ValueError("Select one member for Ertel PV.")
        pressure = snap.level.values.astype(float) * 100.0
        if len(pressure) != 3 or np.any(np.diff(pressure) <= 0):
            raise ValueError("PV needs three distinct, increasing pressure levels.")
        theta = np.asarray(snap.temperature.values, dtype=float) * (100000.0 / pressure[:, None, None]) ** KAPPA
        theta_p = np.gradient(theta, pressure, axis=0, edge_order=2)
        u_p = np.gradient(u, pressure, axis=0, edge_order=2)
        v_p = np.gradient(v, pressure, axis=0, edge_order=2)
        theta_x = np.gradient(theta, lam, axis=-1, edge_order=2) / (EARTH_RADIUS_M * cosphi)
        theta_y = np.gradient(theta, phi, axis=-2, edge_order=2) / EARTH_RADIUS_M
        zeta = (np.gradient(v, lam, axis=-1, edge_order=2)
                - np.gradient(u * cosphi, phi, axis=-2, edge_order=2)) / (EARTH_RADIUS_M * cosphi)
        coriolis = 2 * EARTH_OMEGA_S1 * np.sin(phi)[:, None]
        pv = -STANDARD_GRAVITY * (u_p * theta_y - v_p * theta_x + (zeta + coriolis) * theta_p)
        index = list(snap.level.values).index(spec.level)
        values = pv[index] * 1e6
        dims = ("lat", "lon")
    else:
        raise ValueError(f"Unknown derived diagnostic {spec.key}.")
    result = xr.Dataset({"value": (dims, values)}, coords={dim: snap[dim] for dim in dims})
    west, east, south, north = domain
    result = result.sel(lat=slice(south, north), lon=slice(west, east)).reset_coords(drop=True)
    result.value.attrs = {"units": spec.units, "long_name": spec.title}
    return result


def unit_kind(units):
    value = str(units).lower().strip().replace(" ", "").replace("**", "^")
    value = value.replace("²", "2").replace("⁻¹", "-1").replace("⁻²", "-2").replace("^", "")
    value = value.replace("(", "").replace(")", "").replace("·", "")
    groups = {
        "geopotential": {"m2/s2", "m2s-2"}, "height": {"m", "gpm", "geopotentialmetres", "geopotentialmeters"},
        "wind": {"m/s", "ms-1"}, "kelvin": {"k", "kelvin"}, "celsius": {"degc", "celsius", "°c"},
        "humidity": {"kg/kg", "kgkg-1", "1"}, "pa": {"pa"}, "hpa": {"hpa", "mbar", "mb"},
        "vertical_velocity": {"pa/s", "pas-1"}, "millimeters": {"mm"},
    }
    return next((kind for kind, choices in groups.items() if value in choices), None)


def convert_field(snap, spec):
    def field(name, accepted):
        result = snap[name]
        kind = unit_kind(result.attrs.get("units", ""))
        if kind not in accepted:
            raise ValueError(f"Unexpected/missing units for {name}: {result.attrs.get('units')!r}. Check schema before converting.")
        return result, kind

    if spec.conversion == "wind":
        u, _ = field(spec.variables[0], {"wind"})
        v, _ = field(spec.variables[1], {"wind"})
        result = xr.Dataset({"value": np.hypot(u, v), "u": u, "v": v})
    else:
        accepted = {"height": {"geopotential", "height"}, "temperature": {"kelvin", "celsius"},
                    "humidity": {"humidity"}, "pressure": {"pa", "hpa"},
                    "precipitation": {"height", "millimeters"}, "identity": {"vertical_velocity"}}
        value, kind = field(spec.variables[0], accepted[spec.conversion])
        if spec.conversion == "height" and kind == "geopotential":
            value = value / 9.80665
        elif spec.conversion == "temperature" and kind == "kelvin":
            value = value - 273.15
        elif spec.conversion == "humidity":
            value = value * 1000
        elif spec.conversion == "pressure" and kind == "pa":
            value = value / 100
        elif spec.conversion == "precipitation" and kind == "height":
            value = value * 1000
        result = xr.Dataset({"value": value})
    # Cache only spatial fields, not scalar time/level coordinates that differ per row.
    result = result.reset_coords(drop=True)
    result["value"].attrs = {"units": spec.units, "long_name": spec.title}
    if "u" in result:
        result.u.attrs = result.v.attrs = {"units": "m s-1"}
    return result


def cache_identity(store, init_time, lead, member, spec, domain, diagnostic=None):
    identity = {"schema": SCHEMA_VERSION, "store": store, "init_time": iso_utc(init_time),
            "lead_hours": int(lead), "member": str(member), "field": spec.key,
            "variables": list(spec.variables), "level_hpa": spec.level,
            "conversion": spec.conversion, "domain": list(domain)}
    if diagnostic is not None:
        identity["diagnostic"] = diagnostic
    return identity


def load_panel(archive, cache_dir, request, member, spec, domain):
    store, ds = archive.run(request["init_time"])
    diagnostic = diagnostic_settings(ds, spec, domain)
    identity = cache_identity(store, request["init_time"], request["lead_hours"], member, spec, domain, diagnostic)
    encoded = json.dumps(identity, sort_keys=True)
    digest = hashlib.sha256(encoded.encode()).hexdigest()[:24]
    path = Path(cache_dir) / f"wn2_{digest}.nc"
    if path.exists():
        with xr.open_dataset(path) as cached:
            if cached.attrs.get("identity") != encoded:
                raise ValueError(f"Cache identity mismatch: {path}.")
            result = cached.load()
        hit = True
    else:
        selection_domain = diagnostic_domain(ds, domain) if diagnostic else domain
        stencil = diagnostic.get("pressure_stencil_hpa") if diagnostic else None
        snap = select_forecast(ds, request["init_time"], int(request["lead_hours"]), member,
                               spec.variables, spec.level, selection_domain, level_stencil=stencil)
        # Historical chunks include 4 members and all 13 levels over the globe.
        # Decode chunks serially to limit RAM; only the selected regional slice is retained.
        result = derive_field(snap, spec, domain) if diagnostic else convert_field(snap, spec).load(scheduler="synchronous")
        if not np.isfinite(result.value.values).any():
            raise ValueError("Selected forecast field has no finite data in this region.")
        result.attrs["identity"] = encoded
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(".tmp.nc")
        result.to_netcdf(temporary)
        temporary.replace(path)
        hit = False
    record = {**identity, "event_number": int(request["event_number"]), "review_key": request["review_key"],
              "event_start": iso_utc(request["event_start"]), "event_end": iso_utc(request["event_end"]),
              "event_peak": iso_utc(request["event_peak"]), "valid_time": iso_utc(request["valid_time"]),
              "onset_offset_hours": float(request["onset_offset_hours"]), "units": spec.units,
              "field_title": spec.title, "display_min": spec.limits[0], "display_max": spec.limits[1],
              "palette": spec.palette, "cache_hit": hit, "cache_file": str(path), "status": "complete",
              "source_layout": ds.attrs.get("source_layout", "documented_single_run_schema"),
              "source_variable_units": json.dumps({name: {"units": ds[name].attrs.get("units"),
                  "units_source": ds[name].attrs.get("units_source", "source variable attributes")} for name in spec.variables}, sort_keys=True),
              "plot_style_version": PLOT_STYLE_VERSION,
              "wind_vector_spacing_degrees": 1 if domain[1] - domain[0] <= 25 else 2,
              "wind_quiver_scale": 350, "wind_key_ms": 10,
              "height_contour_interval_m": (60 if spec.level is not None and spec.level <= 500 else 30) if spec.conversion == "height" else None,
              "training_cutoff_status": "operational archive training cutoff not verified"}
    return {"data": result, "metadata": record, "spec": spec, "domain": tuple(domain)}


def event_panels(archive, cache_dir, row, member, specs, domains, progress=print):
    if len(specs) != 2 or len(domains) != 2:
        raise ValueError("Choose exactly two plot rows.")
    requests = forecast_requests(row)
    panels = []
    for row_index, (spec, domain) in enumerate(zip(specs, domains)):
        for request in requests.to_dict("records"):
            request.update(event_number=int(row.event_number), review_key=row.review_key,
                           event_end=row.event_end, event_peak=row.event_peak)
            progress(f"Row {row_index + 1} | {spec.title} | {request['lead_days']}-day lead | member {member}")
            panel = load_panel(archive, cache_dir, request, member, spec, domain)
            panel["metadata"].update(vortex_label_metadata(row))
            panel["metadata"]["row_index"] = row_index
            panels.append(panel)
    verify_six_panels(panels)
    return panels


def verify_six_panels(panels):
    if len(panels) != 6:
        raise ValueError("A complete event needs exactly six panels.")
    for key in ("member", "valid_time", "review_key"):
        if len({p["metadata"][key] for p in panels}) != 1:
            raise ValueError(f"Six-panel group mixes {key} values.")
    for row in (0, 1):
        group = [p for p in panels if p["metadata"]["row_index"] == row]
        if sorted(p["metadata"]["lead_hours"] for p in group) != list(LEAD_HOURS):
            raise ValueError("Each row must contain exactly the 1-, 5-, and 10-day leads.")
        if len({json.dumps([p["spec"].key, p["spec"].level, p["spec"].limits, p["domain"]]) for p in group}) != 1:
            raise ValueError("Plot settings differ within a row.")
    for panel in panels:
        m = panel["metadata"]
        if utc_naive(m["init_time"]) + pd.Timedelta(hours=m["lead_hours"]) != utc_naive(m["valid_time"]):
            raise ValueError("Initialization + forecast lead does not equal valid time.")


def prepare_map_assets():
    """Resolve Natural Earth files once, before drawing, so network errors are clear."""
    import cartopy.io.shapereader as shpreader
    for category, name in (("physical", "coastline"), ("cultural", "admin_0_boundary_lines_land")):
        shpreader.natural_earth(resolution="110m", category=category, name=name)


def draw_panel(ax, panel, features=True):
    from matplotlib.colors import TwoSlopeNorm
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    spec, ds, metadata = panel["spec"], panel["data"], panel["metadata"]
    crs = ccrs.PlateCarree()
    ax.set_extent(panel["domain"], crs=crs)
    low, high = spec.limits
    if spec.key in DERIVED_FIELDS:
        if not low < 0 < high:
            raise ValueError("Divergence/PV color scales must include negative values, zero, and positive values.")
        levels = np.r_[np.linspace(low, 0, 7), np.linspace(0, high, 7)[1:]]
        norm = TwoSlopeNorm(vmin=low, vcenter=0, vmax=high)
    else:
        levels, norm = np.linspace(low, high, 13), None
    shade = ax.contourf(ds.lon.values, ds.lat.values, ds.value.values, levels=levels,
                        cmap=spec.palette, norm=norm, extend="both", transform=crs)
    if spec.conversion == "height":
        interval = 60 if spec.level is not None and spec.level <= 500 else 30
        lo, hi = float(ds.value.min()), float(ds.value.max())
        contours = np.arange(np.floor(lo / interval) * interval, hi + interval, interval)
        if len(contours) > 1 and hi > lo:
            cs = ax.contour(ds.lon.values, ds.lat.values, ds.value.values,
                            levels=contours, colors="0.2", linewidths=.6, transform=crs)
            ax.clabel(cs, fontsize=7, fmt="%.0f", inline=True)
    if "u" in ds:
        # Regional winds get 1° vectors; the larger circulation map gets 2°.
        spacing = metadata["wind_vector_spacing_degrees"]
        lat_step = max(1, int(round(spacing / abs(float(ds.lat[1] - ds.lat[0])))))
        lon_step = max(1, int(round(spacing / abs(float(ds.lon[1] - ds.lon[0])))))
        sub = ds.isel(lat=slice(None, None, lat_step), lon=slice(None, None, lon_step))
        arrows = ax.quiver(sub.lon.values, sub.lat.values, sub.u.values, sub.v.values,
                           transform=crs, angles="uv", scale=metadata["wind_quiver_scale"], width=.003)
        ax.quiverkey(arrows, .82, -.10, metadata["wind_key_ms"], "10 m s⁻¹", labelpos="E", coordinates="axes", fontproperties={"size": 8})
    if features:
        ax.coastlines(resolution="110m", linewidth=.7)
        ax.add_feature(cfeature.BORDERS.with_scale("110m"), linewidth=.4, edgecolor=".4")
    for name, vertices in POLYGONS.items():
        xy = np.asarray(vertices + [vertices[0]])
        ax.plot(xy[:, 0], xy[:, 1], color="darkmagenta" if name == "JPCZ polygon" else "darkorange",
                linewidth=1.2, transform=crs, label=name)
    grid = ax.gridlines(draw_labels=True, linewidth=.35, color=".4", alpha=.5)
    grid.top_labels = grid.right_labels = False
    grid.xlabel_style = grid.ylabel_style = {"size": 8}
    title = (f"Observed event: {vortex_plot_label(metadata)}\n"
                 f"{spec.title} | member {metadata['member']} | {metadata['lead_hours'] // 24}-day lead\n"
                 f"Init: {utc_naive(metadata['init_time']):%Y-%m-%d %H:%M} UTC\n"
                 f"Valid: {utc_naive(metadata['valid_time']):%Y-%m-%d %H:%M} UTC")
    if spec.key == "ertel_pv":
        method = metadata["diagnostic"]["vertical_derivative"]
        stencil = "/".join(map(str, metadata["diagnostic"]["pressure_stencil_hpa"]))
        title += f"\nPV stencil {stencil} hPa · {'one-sided' if method.startswith('one-sided') else 'centered'}"
    ax.set_title(title, fontsize=9, pad=9)
    return shade


def colorbar_label(spec):
    return "10⁻⁵ s⁻¹\nblue: convergence; red: divergence" if spec.key == "divergence" else spec.units


def render_event(panels, row, features=True):
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    verify_six_panels(panels)
    fig, axes = plt.subplots(2, 3, figsize=(17, 10.8), subplot_kw={"projection": ccrs.PlateCarree()})
    fig.subplots_adjust(left=.045, right=.91, top=.80, bottom=.12, hspace=.52, wspace=.22)
    for index, panel in enumerate(panels):
        artist = draw_panel(axes.flat[index], panel, features)
        if index % 3 == 2:
            position = axes.flat[index].get_position()
            cax = fig.add_axes([.935, position.y0, .012, position.height])
            fig.colorbar(artist, cax=cax, label=colorbar_label(panel["spec"]))
    m = panels[0]["metadata"]
    peak = pd.to_numeric(row.get("event_peak_D_1e5_s-1", np.nan), errors="coerce")
    fig.suptitle(f"WeatherNext 2 | JPCZ event {int(row.event_number)} | Observed event: {vortex_plot_label(row)} | ensemble member {m['member']}\n"
                 f"Catalog start: {utc_naive(row.event_start):%Y-%m-%d %H:%M} UTC  |  "
                 f"end: {utc_naive(row.event_end):%Y-%m-%d %H:%M} UTC\n"
                 f"Target: {utc_naive(m['valid_time']):%Y-%m-%d %H:%M} UTC  "
                 f"(target − onset: {m['onset_offset_hours']:+g} h)  |  "
                 f"Peak: {utc_naive(row.event_peak):%Y-%m-%d %H:%M} UTC; D12 = {peak:.2f} × 10⁻⁵ s⁻¹",
                 fontsize=12, y=.985, linespacing=1.6)
    axes[0, 0].legend(loc="lower left", fontsize=7, framealpha=.85)
    fig.text(.05, .045, "1 / 5 / 10 days = exact forecast leads to the same target valid time. "
             "Magenta: JPCZ polygon; orange: coastal wedge.\n"
             "Study window is provisional: operational WN2 training cutoff unverified. "
             "Pressure-level fields/diagnostics are unmasked below the surface (especially 1000 hPa).\n"
             "Source: Google DeepMind / Google Research WeatherNext 2 experimental historical data (CC BY 4.0).",
             fontsize=8, linespacing=1.7)
    return fig


def export_event(panels, row, output_dir, figure=None, features=True):
    """Six standalone panels plus the overview, metadata, settings, and attribution."""
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    verify_six_panels(panels)
    metadata = [p["metadata"] for p in panels]
    # Include plotting settings and event identity; cache-hit status is irrelevant.
    stable = [{k: v for k, v in m.items() if k not in ("cache_hit", "cache_file")} for m in metadata]
    signature = hashlib.sha256(json.dumps(stable, sort_keys=True).encode()).hexdigest()[:12]
    m = metadata[0]
    safe_member = re.sub(r"[^A-Za-z0-9_.-]", "_", m["member"])
    destination = Path(output_dir) / (f"event_{int(row.event_number):04d}_start{utc_naive(row.event_start):%Y%m%dT%H%M}"
                                    f"_member{safe_member}_{signature}")
    destination.mkdir(parents=True, exist_ok=True)
    required = [destination / f"overview.{ext}" for ext in ("png", "pdf")]
    for panel in panels:
        item = panel["metadata"]
        stem = f"row{item['row_index'] + 1}_{panel['spec'].key}_lead{item['lead_hours']:03d}h"
        required.extend(destination / f"{stem}.{ext}" for ext in ("png", "pdf"))
    required.extend(destination / name for name in ("forecast_metadata.csv", "settings.json", "attribution.txt"))
    marker = destination / "complete.json"
    if marker.exists() and all(path.exists() and path.stat().st_size > 0 for path in required):
        saved = json.loads(marker.read_text())
        if saved.get("signature") == signature:
            return destination
    marker.unlink(missing_ok=True)
    own_figure = figure is None
    figure = render_event(panels, row, features) if own_figure else figure
    for extension in ("png", "pdf"):
        figure.savefig(destination / f"overview.{extension}", dpi=180, bbox_inches="tight")
    if own_figure:
        plt.close(figure)
    for panel in panels:
        item = panel["metadata"]
        fig, ax = plt.subplots(figsize=(8, 6.8), subplot_kw={"projection": ccrs.PlateCarree()})
        fig.subplots_adjust(bottom=.20, top=.80)
        artist = draw_panel(ax, panel, features)
        fig.colorbar(artist, ax=ax, shrink=.8, pad=.08, label=colorbar_label(panel["spec"]))
        fig.suptitle(f"JPCZ event {int(row.event_number)} | WN2 ensemble member {item['member']}\n"
                     f"Onset {utc_naive(row.event_start):%Y-%m-%d %H:%M} UTC; target − onset {item['onset_offset_hours']:+g} h", fontsize=11)
        fig.text(.08, .04, "Google DeepMind / Google Research WN2 historical data, CC BY 4.0.\n"
                 "Magenta: JPCZ polygon; orange: coastal wedge. Pressure-level fields/diagnostics are unmasked below the surface.\n"
                 "Operational model training cutoff unverified; see metadata and attribution.txt.", fontsize=7)
        stem = f"row{item['row_index'] + 1}_{panel['spec'].key}_lead{item['lead_hours']:03d}h"
        for extension in ("png", "pdf"):
            fig.savefig(destination / f"{stem}.{extension}", dpi=180, bbox_inches="tight")
        plt.close(fig)
    pd.DataFrame(metadata).to_csv(destination / "forecast_metadata.csv", index=False)
    (destination / "settings.json").write_text(json.dumps(stable, indent=2, ensure_ascii=False))
    (destination / "attribution.txt").write_text(
        "© 2025 DeepMind Technologies Limited's machine learning models used to create the experimental data made available at "
        "https://developers.google.com/earth-engine/datasets/catalog/projects_gcp-public-data-weathernext_assets_weathernext_2_0_0 "
        "under CC BY 4.0 licence terms. This data is intended for experimental modelling only and is not intended, validated, "
        "or approved for real world use.\n\nOperational WN2 model training cutoff has not been verified.\n")
    # A complete marker is written only after all seven figure pairs and metadata exist.
    (destination / "complete.json").write_text(json.dumps({"signature": signature, "forecast_panels": 6, "member": m["member"]}))
    return destination


def find_notebook35_references(review_dir, row):
    records = []
    for path in sorted(Path(review_dir).glob("*_metadata.csv")):
        try:
            saved = pd.read_csv(path)
            if not all(c in saved for c in DATE_COLUMNS):
                continue
            match = pd.Series(True, index=saved.index)
            for column in DATE_COLUMNS:
                match &= pd.to_datetime(saved[column], utc=True, errors="coerce").dt.tz_localize(None).eq(utc_naive(row[column]))
            image = path.with_name(path.name.removesuffix("_metadata.csv") + ".png")
            for item in saved.loc[match].to_dict("records"):
                current = vortex_plot_label(row)
                matches_label = item.get("label_scheme") == VORTEX_LABEL_SCHEME and item.get("label") == current
                records.append({**item, "current_vortex_classification": current, "reference_label_current": matches_label,
                                "metadata_file": str(path), "image_file": str(image) if image.exists() else ""})
        except (ValueError, OSError, pd.errors.ParserError):
            continue
    result = pd.DataFrame(records)
    return result.sort_values("reference_label_current", ascending=False, kind="stable") if len(result) else result


In [ ]:
# @title 3c · Load verification helpers (run unchanged)
"""Onset-aligned verification of WN2 against ERA5 for the two JPCZ regions."""

from __future__ import annotations

from functools import lru_cache
import hashlib
import json
from pathlib import Path
from tempfile import NamedTemporaryFile

import numpy as np
import pandas as pd
import xarray as xr



ERA5_STORE = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
VERIFICATION_DOMAIN = (125., 145., 33., 47.)
ENSEMBLE_SIZE = 64
DIVERGENCE_SPEC = FIELD_PRESETS["divergence"]
DIVERGENCE_COLUMN = "divergence_1e5_s-1"
RANK_GROUPS = ("1–13", "14–26", "27–39", "40–52", "53–65")
CHECKPOINT_VERSION = 1


def polygon_divergence(field):
    """Area-weighted means on the native grid, independently for each polygon."""
    from matplotlib.path import Path as PolygonPath

    lon, lat = np.meshgrid(field.lon.values, field.lat.values)
    points = np.column_stack((lon.ravel(), lat.ravel()))
    means, counts = {}, {}
    for region, vertices in POLYGONS.items():
        mask = PolygonPath(vertices).contains_points(points).reshape(lon.shape)
        weights = np.cos(np.deg2rad(lat[mask]))
        values = field.values[..., mask]
        if not mask.any() or not np.isfinite(values).all():
            raise ValueError(f"{region} lacks a complete finite set of divergence cells.")
        means[region] = np.average(values, weights=weights, axis=-1)
        counts[region] = int(mask.sum())
    return means, counts


@lru_cache(maxsize=1)
def open_verification_era5():
    return xr.open_zarr(ERA5_STORE, chunks={}, storage_options={"token": "anon"})


def era5_polygon_values(valid_time, era5):
    """Derive divergence from the exact ERA5 925-hPa wind snapshot."""
    target = utc_naive(valid_time)
    winds = era5[list(DIVERGENCE_SPEC.variables)].sel(time=target, level=DIVERGENCE_SPEC.level)
    winds = winds.rename({"latitude": "lat", "longitude": "lon"})
    west, east, south, north = diagnostic_domain(winds, VERIFICATION_DOMAIN)
    winds = winds.sortby("lat").sortby("lon").sel(lat=slice(south, north), lon=slice(west, east))
    if any(abs(actual - expected) > 1e-5 for actual, expected in zip(
            (float(winds.lon.min()), float(winds.lon.max()), float(winds.lat.min()), float(winds.lat.max())),
            (west, east, south, north))):
        raise ValueError("ERA5 grid does not cover the verification domain and its derivative halo.")
    winds = winds.transpose("lat", "lon")
    field = derive_field(winds, DIVERGENCE_SPEC, VERIFICATION_DOMAIN).value
    means, counts = polygon_divergence(field)
    return {"valid_time": iso_utc(target), "source_store": ERA5_STORE, "polygon_cells": counts,
            "values": {region: float(value) for region, value in means.items()}}


def forecast_polygon_values(archive, request, progress=print):
    """Read groups of native WN2 samples and immediately reduce each field to two means."""
    store, ds = archive.run(request["init_time"])
    labels = [str(value) for value in ds.sample.values.tolist()]
    if len(labels) != ENSEMBLE_SIZE or len(set(labels)) != ENSEMBLE_SIZE:
        raise ValueError(f"Expected {ENSEMBLE_SIZE} distinct WN2 member labels, found {len(set(labels))}.")
    snap = select_forecast(
        ds, request["init_time"], request["lead_hours"], None, DIVERGENCE_SPEC.variables,
        DIVERGENCE_SPEC.level, diagnostic_domain(ds, VERIFICATION_DOMAIN),
    )
    winds = snap[DIVERGENCE_SPEC.variables[0]]
    sizes = winds.chunksizes.get("sample", (4,) * (ENSEMBLE_SIZE // 4))
    values = {region: [] for region in POLYGONS}
    start = 0
    for size in sizes:
        end = start + size
        progress(f"  Reading {request['lead_days']}-day members {start + 1}–{end} of {ENSEMBLE_SIZE}")
        block = snap.isel(sample=slice(start, end))
        field = derive_field(block, DIVERGENCE_SPEC, VERIFICATION_DOMAIN).value
        means, counts = polygon_divergence(field)
        for region in POLYGONS:
            values[region].extend(means[region].tolist())
        start = end
    return {"init_time": iso_utc(request["init_time"]), "valid_time": iso_utc(request["valid_time"]),
            "lead_hours": request["lead_hours"], "source_store": store, "members": labels,
            "polygon_cells": counts, "values": values}


def checkpoint_identity(archive):
    """Reuse verified values only with the same fields, polygons and source configuration."""
    return {
        "version": CHECKPOINT_VERSION, "era5_store": ERA5_STORE, "wn2_root": archive.root,
        "wn2_overrides": archive.overrides, "domain": VERIFICATION_DOMAIN, "polygons": POLYGONS,
        "pressure_hpa": DIVERGENCE_SPEC.level, "ensemble_size": ENSEMBLE_SIZE,
        "diagnostic": diagnostic_settings(None, DIVERGENCE_SPEC, VERIFICATION_DOMAIN),
    }


def atomic_json_file(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent, delete=False) as temporary:
        temporary_path = Path(temporary.name)
        try:
            json.dump(payload, temporary, allow_nan=False, indent=2)
        except Exception:
            temporary.close()
            temporary_path.unlink(missing_ok=True)
            raise
    temporary_path.replace(path)


def checkpoint(path, identity, calculate):
    if path.is_file():
        try:
            saved = json.loads(path.read_text())
            if saved["request"] == identity:
                return saved["data"], True
        except (KeyError, ValueError, TypeError):
            pass
    data = calculate()
    atomic_json_file(path, {"request": identity, "data": data})
    return data, False


def observation_rank(members, reference, case_key, *, tie_seed=37):
    """Return the ERA5 rank, its tied interval, and its position relative to the ensemble."""
    values = np.asarray(members, dtype=np.float64)
    if values.shape != (ENSEMBLE_SIZE,) or not np.isfinite(values).all() or not np.isfinite(reference):
        raise ValueError("Ranking requires 64 finite member values and one finite ERA5 value.")
    less = int(np.count_nonzero(values < reference))
    equal = int(np.count_nonzero(values == reference))
    lower, upper = less + 1, less + equal + 1
    key = f"{tie_seed}|{case_key}".encode()
    rng = np.random.default_rng(int.from_bytes(hashlib.sha256(key).digest()[:8], "big"))
    rank = lower if lower == upper else int(rng.integers(lower, upper + 1))
    return {
        "era5_rank": rank, "rank_min_if_tied": lower, "rank_max_if_tied": upper,
        "rank_group": RANK_GROUPS[(rank - 1) // 13],
        "below_member_min": bool(reference < values.min()),
        "above_member_max": bool(reference > values.max()),
    }


def case_identity(row):
    return {
        "event_number": int(row.event_number), "review_key": row.review_key,
        "event_start_utc": iso_utc(row.event_start), "event_end_utc": iso_utc(row.event_end),
        "event_peak_utc": iso_utc(row.event_peak), "valid_time_utc": iso_utc(row.target_valid_time),
        "target_minus_onset_hours": float(row.onset_offset_hours),
        "catalog_peak_D12_1e5_s-1": row["event_peak_D_1e5_s-1"],
        "vortex_label": row.get("vortex_plot_label", ""),
        "vortex_scan_status": row.get("vortex_scan_status", ""),
    }


def run_ensemble_verification(catalog, archive, output_dir, *, era5_reader=open_verification_era5, progress=print):
    """Checkpoint one ERA5 target and each WN2 run/lead independently."""
    identity = checkpoint_identity(archive)
    signature = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
    cache = Path(output_dir) / "checkpoints" / signature
    member_rows, reference_rows, rank_rows, status_rows = [], [], [], []

    for _, row in catalog.iterrows():
        event = case_identity(row)
        target = utc_naive(row.target_valid_time)
        stamp = target.strftime("%Y%m%dT%H%MZ")
        progress(f"Event {event['event_number']} | verification {event['valid_time_utc']}")
        reference = None
        try:
            reference, cached = checkpoint(
                cache / f"era5_{stamp}.json", {"valid_time": event["valid_time_utc"]},
                lambda: era5_polygon_values(target, era5_reader()),
            )
            reference_rows.extend([
                {**event, "polygon": region, "pressure_hpa": DIVERGENCE_SPEC.level,
                 DIVERGENCE_COLUMN: reference["values"][region],
                 "polygon_cells": reference["polygon_cells"][region], "source_store": reference["source_store"]}
                for region in POLYGONS
            ])
            ref_status = "cached" if cached else "computed"
            ref_error = ""
        except Exception as error:
            reference = None
            ref_status = "failed"
            ref_error = f"{type(error).__name__}: {error}"
            progress(f"  ERA5 unavailable: {ref_error}")

        event_ranks = []
        for request in forecast_requests(row).to_dict("records"):
            lead = request["lead_hours"]
            timing = {"lead_hours": lead, "lead_days": request["lead_days"], "init_time_utc": iso_utc(request["init_time"])}
            status = {**event, **timing, "era5_status": ref_status, "era5_error": ref_error}
            try:
                forecast, cached = checkpoint(
                    cache / f"wn2_{stamp}_lead{lead:03d}h.json",
                    {"init_time": timing["init_time_utc"], "lead_hours": lead, "valid_time": event["valid_time_utc"]},
                    lambda: forecast_polygon_values(archive, request, progress),
                )
                members = forecast["members"]
                if len(members) != ENSEMBLE_SIZE or len(set(members)) != ENSEMBLE_SIZE:
                    raise ValueError("Forecast checkpoint does not contain 64 distinct members.")
                case_values, case_ranks = [], []
                for region in POLYGONS:
                    values = np.asarray(forecast["values"][region], dtype=float)
                    if values.shape != (ENSEMBLE_SIZE,) or not np.isfinite(values).all():
                        raise ValueError(f"{region} does not contain 64 finite member values.")
                    info = {**event, **timing, "polygon": region, "pressure_hpa": DIVERGENCE_SPEC.level}
                    case_values.extend({**info, "member": member, DIVERGENCE_COLUMN: value,
                                        "polygon_cells": forecast["polygon_cells"][region],
                                        "source_store": forecast["source_store"]} for member, value in zip(members, values))
                    if reference is not None:
                        observed = reference["values"][region]
                        rank = observation_rank(values, observed, f"{row.review_key}|{lead}|{region}")
                        case_ranks.append({**info, "member_count": len(members),
                                           "era5_divergence_1e5_s-1": observed,
                                           "member_min_1e5_s-1": values.min(), "member_mean_1e5_s-1": values.mean(),
                                           "member_max_1e5_s-1": values.max(), **rank})
                member_rows.extend(case_values)
                event_ranks.extend(case_ranks)
                status.update(wn2_status="cached" if cached else "computed", wn2_error="")
            except Exception as error:
                message = f"{type(error).__name__}: {error}"
                status.update(wn2_status="failed", wn2_error=message)
                progress(f"  {request['lead_days']}-day forecast unavailable: {message}")
            status_rows.append(status)
        common_sample = len(event_ranks) == len(LEAD_HOURS) * len(POLYGONS)
        for record in event_ranks:
            record["in_common_sample"] = common_sample
        rank_rows.extend(event_ranks)
    return {
        "case_ranks": pd.DataFrame(rank_rows),
        "member_values": pd.DataFrame(member_rows),
        "era5_reference": pd.DataFrame(reference_rows),
        "batch_status": pd.DataFrame(status_rows),
    }, identity


def plot_rank_histograms(ranks):
    """Use five equally sized groups of ranks, with the same cases at each lead."""
    import matplotlib.pyplot as plt

    if ranks.empty or not ranks.in_common_sample.any():
        raise ValueError("No event has complete ERA5 and all three WN2 leads.")
    complete = ranks.loc[ranks.in_common_sample]
    n_events = complete.review_key.nunique()
    largest_bin = complete.groupby(["polygon", "lead_hours", "rank_group"]).size().max()
    fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharey=True, layout="constrained")
    for r, region in enumerate(POLYGONS):
        for c, lead in enumerate(LEAD_HOURS):
            ax = axes[r, c]
            matched = complete.loc[(complete.polygon == region) & (complete.lead_hours == lead)]
            counts = matched.rank_group.value_counts().reindex(RANK_GROUPS, fill_value=0)
            ax.bar(RANK_GROUPS, counts, color="#277486", width=.8)
            ax.axhline(n_events / len(RANK_GROUPS), color=".25", linestyle="--", linewidth=1)
            ax.set_title(f"{region} · {lead // 24}-day lead")
            ax.set_xlabel("ERA5 rank (most to least convergent)")
            ax.text(.03, .96, f"Outside ensemble range: {matched.below_member_min.sum()} below, "
                    f"{matched.above_member_max.sum()} above", transform=ax.transAxes, va="top", fontsize=8)
            ax.set_ylim(0, 1.22 * max(largest_bin, n_events / len(RANK_GROUPS)))
            ax.yaxis.get_major_locator().set_params(integer=True)
            if c == 0:
                ax.set_ylabel("Number of JPCZ cases")
    fig.suptitle(f"925-hPa divergence at JPCZ onset · {n_events} matched {'case' if n_events == 1 else 'cases'}"
                 f"{' (pilot)' if n_events == 1 else ''}\n"
                 "ERA5 ranks among 64 WeatherNext 2 members · 13 ranks per bar · dashed line = uniform ranks")
    return fig


def export_verification(tables, identity, destination):
    """Write the results and a workbook for inspecting the computed ranks in Drive."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    atomic_json_file(destination / "calculation_settings.json", identity)
    for name, frame in tables.items():
        frame.to_csv(destination / f"{name}.csv", index=False)
    with NamedTemporaryFile(suffix=".xlsx", dir=destination, delete=False) as temporary:
        temporary_path = Path(temporary.name)
    try:
        with pd.ExcelWriter(temporary_path, engine="xlsxwriter",
                            engine_kwargs={"options": {"strings_to_formulas": False, "strings_to_urls": False}}) as writer:
            for name, frame in tables.items():
                sheet = name.replace("_", " ").capitalize()
                frame.to_excel(writer, sheet_name=sheet, index=False)
                worksheet = writer.sheets[sheet]
                worksheet.freeze_panes(1, 0)
                if len(frame.columns):
                    worksheet.autofilter(0, 0, len(frame), len(frame.columns) - 1)
                    for index, column in enumerate(frame.columns):
                        width = max(len(column) + 2, 14)
                        if not frame.empty:
                            width = max(width, int(frame[column].head(100).astype(str).str.len().max()) + 2)
                        worksheet.set_column(index, index, min(width, 55))
        temporary_path.replace(destination / "ensemble_verification.xlsx")
    finally:
        temporary_path.unlink(missing_ok=True)
    return destination


In [ ]:
# @title 3d · Preview the selected catalog
from IPython.display import display

catalog, selection_report = read_catalog(
    CATALOG_PATH, NOTEBOOK35_DIR, STUDY_START_UTC, STUDY_END_UTC or None)
if catalog.empty:
    raise ValueError('The current catalog has no events in the configured study window.')
if RUN_MODE not in ('Pilot', 'Catalog'):
    raise ValueError('Choose Pilot or Catalog.')
if RUN_MODE == 'Pilot':
    selected = catalog.head(1) if PILOT_EVENT_NUMBER == 0 else catalog.loc[
        catalog.event_number.eq(PILOT_EVENT_NUMBER)]
    if selected.empty:
        raise ValueError('Pilot event not found in this study window. Pick a listed event number.')
    result_directory = OUTPUT_DIR / f'pilot_event_{int(selected.event_number.iloc[0]):04d}'
else:
    selected = catalog
    result_directory = OUTPUT_DIR / 'catalog'
archive = GoogleCloudArchive(credentials, PROJECT_ID or detected_project or '', overrides=RUN_OVERRIDES)
print(f'{len(catalog)} events in the study window. {RUN_MODE} will process {len(selected)}.')
print('Result folder:', result_directory)
display(catalog[['event_number', 'event_start', 'event_peak', 'target_valid_time',
                 'onset_offset_hours', 'vortex_plot_label']])


## 4 · Run the verification

Both inputs use the same spherical divergence calculation with a two-grid-cell halo,
and the same cosine-latitude weighting and polygon boundaries on their native grids.
The fixed 925-hPa fields include any below-ground pressure-level values supplied by the
datasets. The coastal wedge results should be interpreted with that in mind. The notebook
reads WN2 members in source chunk groups and saves two means per member and lead.

Each ERA5 rank is one plus the number of member values below ERA5, with seeded random
tie-breaking. Rank 1 means ERA5 was at the most convergent end and rank 65 at the least
convergent end. The exact out-of-range flags are recorded separately from the rank.

The batch status records missing or failed inputs. Rerunning retries those inputs and
reuses completed checkpoints. Cases with ERA5 and all three forecast leads contribute
to the six histograms so that the lead comparison uses the same events.


In [ ]:
# @title 4 · Read the selected forecasts and calculate ranks
tables, calculation_settings = run_ensemble_verification(selected, archive, OUTPUT_DIR)
display(tables['batch_status'][['event_number', 'lead_days', 'era5_status', 'wn2_status',
                                'era5_error', 'wn2_error']])
ranks = tables['case_ranks']
if not ranks.empty:
    display(ranks[['event_number', 'valid_time_utc', 'polygon', 'lead_days', 'member_count',
                   'era5_divergence_1e5_s-1', 'era5_rank', 'in_common_sample']])


## 5 · Save and inspect results

The Excel workbook opens in Excel or Google Sheets. The same tables are available as
CSVs. The plots save as PNG and PDF. Pilots and the catalog have separate result folders.

Each histogram bar groups 13 of the 65 possible ranks. The dashed line shows the
expected counts for a uniform rank distribution. Pilot histograms are a check of the
workflow. Catalog histograms describe cases selected around ERA5-detected JPCZ events.


In [ ]:
# @title 5 · Save workbook, tables, and six-panel rank histogram
import matplotlib.pyplot as plt

tables['event_selection'] = selection_report
calculation_settings = {
    **calculation_settings, 'catalog': str(CATALOG_PATH), 'study_start_utc': STUDY_START_UTC,
    'study_end_utc_exclusive': STUDY_END_UTC or None, 'run_mode': RUN_MODE,
    'selected_events': len(selected), 'lead_hours': list(LEAD_HOURS), 'tie_seed': 37,
    'training_cutoff_status': 'operational archive training cutoff unverified',
    'wn2_data_reference': (
        'Google LLC WeatherNext 2 historic experimental data (CC BY 4.0). '
        'https://developers.google.com/earth-engine/datasets/catalog/'
        'projects_gcp-public-data-weathernext_assets_weathernext_2_0_0'
    ),
}
export_verification(tables, calculation_settings, result_directory)
print('Saved Excel workbook and CSV tables to', result_directory)
if not ranks.empty and ranks.in_common_sample.any():
    figure = plot_rank_histograms(ranks)
    for extension in ('png', 'pdf'):
        figure.savefig(result_directory / f'rank_histograms.{extension}', dpi=180)
    plt.show()
else:
    print('No complete event for all three leads. Check Batch status, then retry section 4.')


## Follow-up · Reliability diagrams

Reliability will compare a forecast probability of crossing a specified convergence
threshold with a binary ERA5 outcome. Use independently scheduled winter times that
include crossings and non-crossings. The ERA5 catalog detector uses a 12-hour average
of hourly values. Before applying its criterion to WN2, check how well a six-hourly
approximation reproduces the hourly ERA5 decisions. The rank workflow above uses a
matched snapshot and has no convergence threshold.

Forecast data: Google WeatherNext 2 historical experimental data, CC BY 4.0.
Reference: ERA5 reanalysis via Google Research ARCO-ERA5.
